# C2.4 — Advanced AI Workflows: LCEL · LlamaIndex · MCP · Multi-Agent · Prompt Versioning

**Domain focus:** Finance, Education, Healthcare, E-commerce — examples throughout are drawn from these industries.

| Section | Topic |
|---------|-------|
| 1 | LangChain LCEL — multi-step chains and memory |
| 2 | LlamaIndex — data connectors, index types, query engines |
| 3 | MCP — Model Context Protocol with a local server |
| 4 | Multi-agent orchestration — coordinator + specialist |
| 5 | Prompt versioning and A/B testing |
| 6 | Lab — end-to-end AI workflow |

## Setup

In [ ]:
# Install required packages (run once)
# !pip install langchain-core langchain-anthropic llama-index-core llama-index-llms-anthropic
# !pip install llama-index-embeddings-huggingface mcp anthropic

In [ ]:
import os, json, uuid, time, asyncio, subprocess, sys, textwrap
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field
from typing import Optional

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
print('API key loaded:', bool(ANTHROPIC_API_KEY))

---
## Section 1 · LangChain LCEL (LangChain Expression Language)

<small>
LCEL is the **declarative composition layer** introduced in LangChain v0.1+.
It replaces the legacy `LLMChain` / `SequentialChain` classes with a simple pipe syntax.

```
# Legacy (avoid)
chain = LLMChain(llm=llm, prompt=prompt)

# LCEL (current)
chain = prompt | llm | parser
```

**Why LCEL?**
- Streaming and async work out of the box
- Parallel branches with `RunnableParallel`
- Explicit, composable memory via `RunnableWithMessageHistory`
- No hidden state — every step is a Runnable you can inspect
</small>

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(
    model='claude-haiku-4-5-20251001',
    temperature=0.3,
    anthropic_api_key=ANTHROPIC_API_KEY
)
parser = StrOutputParser()
print('LLM ready')

### 1.1 Basic LCEL Chain — Finance: Earnings Report Analysis

<small>A simple three-step chain: prompt → LLM → parser.
The finance scenario: an analyst pastes a raw earnings paragraph; the chain returns a structured summary.
</small>

In [ ]:
earnings_prompt = ChatPromptTemplate.from_template(
    'You are a financial analyst assistant.\n'
    'Analyze the following earnings report excerpt and return:\n'
    '1. Revenue trend (up/down/flat)\n'
    '2. Key risk factors (max 2 bullet points)\n'
    '3. One-line investment signal\n\n'
    'Report:\n{report}'
)

earnings_chain = earnings_prompt | llm | parser

sample_report = '''
Q3 2024: Revenue rose 12% YoY to $4.8B, beating consensus by 3%.
Operating margin compressed 80bps to 18.2% due to higher cloud infrastructure costs.
Guidance for Q4 raised to $5.0–5.1B. Management flagged FX headwinds in EMEA.
'''

result = earnings_chain.invoke({'report': sample_report})
print(result)

### 1.2 Multi-Step Chain with `RunnablePassthrough`

<small>
`RunnablePassthrough` forwards the input unchanged while other keys are added.
Pattern: `{'context': retriever, 'question': RunnablePassthrough()} | prompt | llm`

**Education scenario:** A student submits an essay; the chain grades it, then writes personalised feedback.
</small>

In [ ]:
grade_prompt = ChatPromptTemplate.from_template(
    'Grade this student essay on a scale of A–F with one sentence justification.\n'
    'Topic: {topic}\nEssay:\n{essay}'
)

feedback_prompt = ChatPromptTemplate.from_template(
    'The essay received the following grade:\n{grade}\n\n'
    'Write two specific, encouraging improvement suggestions for the student.'
)

grade_chain = grade_prompt | llm | parser

# Chain grade output into feedback as 'grade' key
full_grading_chain = (
    RunnablePassthrough.assign(grade=grade_chain)
    | feedback_prompt | llm | parser
)

result = full_grading_chain.invoke({
    'topic': 'The impact of AI on modern education',
    'essay': ('AI is transforming classrooms. Students can now access personalised tutors '
              'at any time. However, concerns about academic integrity and over-reliance '
              'on automation remain significant challenges for educators.')
})
print(result)

### 1.3 Parallel Execution — Healthcare: Multi-Angle Patient Report

<small>
`RunnableParallel` runs branches concurrently and collects results as a dict.
Healthcare scenario: analyse a patient discharge summary from three perspectives simultaneously.
</small>

In [ ]:
discharge_summary = '''
Patient: 58-year-old male. Admitted for acute chest pain.
Diagnosed: NSTEMI. Treated with dual antiplatelet therapy.
Discharged after 3 days. Follow-up cardiology appointment in 2 weeks.
Prescribed: aspirin 100mg, ticagrelor 90mg, atorvastatin 40mg.
'''

clinical_prompt   = ChatPromptTemplate.from_template('Summarise the clinical findings:\n{summary}')
medication_prompt = ChatPromptTemplate.from_template('List medication risks to monitor:\n{summary}')
followup_prompt   = ChatPromptTemplate.from_template('What follow-up actions are required?\n{summary}')

parallel_analysis = RunnableParallel(
    clinical_summary  = clinical_prompt   | llm | parser,
    medication_risks  = medication_prompt | llm | parser,
    followup_actions  = followup_prompt   | llm | parser,
)

results = parallel_analysis.invoke({'summary': discharge_summary})
for section, text in results.items():
    print(f'\n=== {section.upper()} ===')
    print(text)

### 1.4 Dynamic Routing — Finance vs Education Queries

<small>
`RunnableBranch` evaluates predicates in order and routes to the first matching branch.
Use this to send finance queries to a finance-tuned prompt and education queries to a different one.
</small>

In [ ]:
finance_prompt = ChatPromptTemplate.from_template(
    'You are a financial advisor. Answer concisely:\n{query}'
)
education_prompt = ChatPromptTemplate.from_template(
    'You are an academic tutor. Answer concisely:\n{query}'
)
healthcare_prompt = ChatPromptTemplate.from_template(
    'You are a medical information assistant. Answer concisely:\n{query}'
)

from langchain_core.runnables import RunnableBranch

router = RunnableBranch(
    (lambda x: any(k in x['query'].lower() for k in ['stock','portfolio','invest','earnings','market']),
     finance_prompt   | llm | parser),
    (lambda x: any(k in x['query'].lower() for k in ['course','study','grade','exam','tutor','student']),
     education_prompt | llm | parser),
    healthcare_prompt | llm | parser,   # default branch
)

queries = [
    {'query': 'How do I diversify my stock portfolio?'},
    {'query': 'What study techniques work best for medical exams?'},
    {'query': 'What are common symptoms of vitamin D deficiency?'},
]

for q in queries:
    print(f'\nQuery : {q["query"]}')
    print(f'Answer: {router.invoke(q)[:200]}')

### 1.5 Memory Management — Education: Adaptive Tutoring Chatbot

<small>
Modern LangChain uses `RunnableWithMessageHistory` instead of deprecated `ConversationBufferMemory`.
Each `session_id` gets its own isolated history — perfect for a multi-student tutoring platform.
</small>

In [ ]:
tutor_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are an adaptive tutor for a finance student. '
               'Remember what the student has told you and build on prior explanations.'),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{input}'),
])

tutor_chain = tutor_prompt | llm

# Session store: production would use Redis or Postgres
session_store = {}

def get_session_history(session_id: str):
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]

tutoring_app = RunnableWithMessageHistory(
    tutor_chain,
    get_session_history,
    input_messages_key='input',
    history_messages_key='history',
)

student_session = 'student_priya_001'

exchanges = [
    'I am a finance student struggling with DCF valuation.',
    'Can you give me a simple example using a coffee shop?',
    'What discount rate should I use for that coffee shop?',
]

for msg in exchanges:
    print(f'\nStudent: {msg}')
    resp = tutoring_app.invoke(
        {'input': msg},
        config={'configurable': {'session_id': student_session}}
    )
    print(f'Tutor  : {resp.content[:300]}')

### 1.6 Streaming — Real-Time Investment Commentary
<small>Any LCEL chain supports streaming with `.stream()` — tokens arrive as they are generated.</small>

In [ ]:
commentary_prompt = ChatPromptTemplate.from_template(
    'Write a 3-sentence market commentary on: {event}'
)
stream_chain = commentary_prompt | llm | parser

print('Streaming commentary:')
for chunk in stream_chain.stream({'event': 'US Federal Reserve holds rates steady'}):
    print(chunk, end='', flush=True)
print()

---
## Section 2 · LlamaIndex — Data Connectors, Index Types, and Query Engines

### When to Use LlamaIndex vs LangChain

| Dimension | LangChain strength | LlamaIndex strength |
|-----------|-------------------|---------------------|
| **Primary use case** | General LLM orchestration, agents, chains | Document retrieval and Q&A |
| **Data ingestion** | Basic loaders, manual chunking | 100+ connectors, smart chunking |
| **Index types** | VectorStore abstraction | Vector, summary, keyword, knowledge graph |
| **Query strategies** | RAG chain (retrieve → generate) | Sub-question, multi-step, router query |
| **Memory** | `RunnableWithMessageHistory`, LangGraph | Chat engine with built-in memory |
| **Agent tools** | LangChain tools, MCP | QueryEngine as a tool |
| **Best for** | Multi-step workflows, routing, production apps | Deep document understanding, RAG |

**Rule of thumb:** Use LangChain when you need complex orchestration logic.
Use LlamaIndex when your bottleneck is retrieval quality over large document sets.
They compose well — a LlamaIndex query engine can be a LangChain tool.

### 2.1 Setup — Finance Document Corpus (In-Memory)
<small>We create synthetic finance and education documents in memory.
In production, use `SimpleDirectoryReader`, `DatabaseReader`, or any of the 100+ LlamaIndex connectors.</small>

In [ ]:
from llama_index.core import Document, VectorStoreIndex, SummaryIndex, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.anthropic import Anthropic as LlamaAnthropic
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.tools import QueryEngineTool

# Configure LlamaIndex to use Claude
Settings.llm = LlamaAnthropic(
    model='claude-haiku-4-5-20251001',
    api_key=ANTHROPIC_API_KEY
)
# Use a small chunk size for the demo corpus
Settings.node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=64)

# ── Finance documents ────────────────────────────────────────────────────────
finance_docs = [
    Document(text='''Annual Report 2023 — FinTech Corp.
Revenue grew 18% to $2.4B. Net income margin improved to 14%.
Digital payments segment contributed 60% of revenue.
Risk factors: regulatory scrutiny in EU, rising cost of capital.
Management targets 20% revenue growth in 2024 through SME expansion.''',
        metadata={'domain': 'finance', 'type': 'annual_report', 'company': 'FinTechCorp'}),

    Document(text='''Q3 2023 Earnings — EduLearn Inc.
Course enrollment rose 35% YoY to 1.2M active learners.
Revenue: $180M (+28% YoY). Gross margin: 72%.
B2B corporate training contracts now represent 40% of revenue.
Guidance: full-year revenue $720M, adjusted EBITDA margin 25%.
Key risk: teacher retention cost increased 15% due to talent competition.''',
        metadata={'domain': 'finance', 'type': 'earnings', 'company': 'EduLearnInc'}),

    Document(text='''Investment Thesis — Healthcare AI Sector 2024.
AI diagnostics market projected to reach $45B by 2027 (CAGR 48%).
Key players: imaging AI, clinical decision support, drug discovery AI.
Regulatory risk: FDA pathway for AI/ML-based software devices tightening.
Recommended allocation: 5–8% of tech portfolio for diversified exposure.''',
        metadata={'domain': 'finance', 'type': 'thesis', 'sector': 'HealthcareAI'}),
]

# ── Education documents ──────────────────────────────────────────────────────
education_docs = [
    Document(text='''Course Catalog 2024 — DataScience Track.
DS-101 Introduction to Python (no prerequisites, 6 weeks).
DS-201 Statistics for Data Science (requires DS-101, 8 weeks).
DS-301 Machine Learning Fundamentals (requires DS-201, 10 weeks).
DS-401 Deep Learning and Neural Networks (requires DS-301, 12 weeks).
Capstone project required for certificate completion.''',
        metadata={'domain': 'education', 'type': 'catalog', 'track': 'DataScience'}),

    Document(text='''Learning Outcomes Report 2023.
Completion rate for DS track: 68% (industry average 45%).
Top reason for dropout: time management (38%), difficulty (29%), career change (18%).
Interventions that improved completion: cohort learning (+12%), weekly check-ins (+9%).
NPS score: 72. Employer satisfaction with graduates: 4.4/5.''',
        metadata={'domain': 'education', 'type': 'outcomes_report'}),
]

all_docs = finance_docs + education_docs
print(f'Loaded {len(all_docs)} documents ({len(finance_docs)} finance, {len(education_docs)} education)')

### 2.2 Index Types

<small>
| Index | How it stores | Best for |
|-------|--------------|---------|
| `VectorStoreIndex` | Embedding vectors | Semantic similarity search |
| `SummaryIndex` | All nodes in sequence | Full-document summarisation |
| `KeywordTableIndex` | Keyword → node mapping | Keyword-based lookup |
</small>

In [ ]:
# VectorStoreIndex — best for semantic Q&A
vector_index = VectorStoreIndex.from_documents(finance_docs)

# SummaryIndex — best for 'give me a summary of everything'
summary_index = SummaryIndex.from_documents(education_docs)

print('Indexes built.')
print(f'VectorIndex nodes  : {len(vector_index.docstore.docs)}')
print(f'SummaryIndex nodes : {len(summary_index.docstore.docs)}')

### 2.3 Query Engines

<small>
A **query engine** wraps an index and exposes a `.query()` method.
The router query engine selects between engines automatically based on the question.
</small>

In [ ]:
# Basic vector query engine
vector_qe = vector_index.as_query_engine(similarity_top_k=2)

# Summary query engine
summary_qe = summary_index.as_query_engine(response_mode='tree_summarize')

# Query the finance corpus
q1 = 'What are the main revenue drivers for FinTech Corp?'
r1 = vector_qe.query(q1)
print(f'Q: {q1}')
print(f'A: {r1}\n')

# Query the education corpus
q2 = 'What interventions improved course completion rates?'
r2 = summary_qe.query(q2)
print(f'Q: {q2}')
print(f'A: {r2}')

### 2.4 Router Query Engine
<small>The router uses an LLM to decide which underlying engine to call for each query.</small>

In [ ]:
# Combine all docs into a single index for router demo
combined_index = VectorStoreIndex.from_documents(all_docs)
combined_qe    = combined_index.as_query_engine(similarity_top_k=3)

# Annotate the engine so the router knows what it covers
finance_tool = QueryEngineTool.from_defaults(
    query_engine=vector_qe,
    description='Finance domain: annual reports, earnings, investment theses for companies.',
)
education_tool = QueryEngineTool.from_defaults(
    query_engine=summary_qe,
    description='Education domain: course catalogs, learning outcomes, completion rates.',
)

router_qe = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[finance_tool, education_tool],
)

for question in [
    'What growth rate is EduLearn targeting?',
    'Which course should I take before DS-301?',
]:
    resp = router_qe.query(question)
    print(f'Q: {question}')
    print(f'A: {resp}\n')

---
## Section 3 · MCP — Model Context Protocol

### What is MCP?

<small>
MCP (Model Context Protocol) is an open standard published by Anthropic that defines how AI models
communicate with external tools and data sources.
It replaces ad-hoc API integrations with a **single protocol** — any MCP server can be used by any MCP client.

```
Client (Claude / LangChain agent)  ←──MCP protocol──→  MCP Server (your tools)
```

**Why MCP matters for multi-agent systems:**
- Agents from different vendors can share the same tool servers
- Tool definitions live in one place — no duplication across agents
- Authentication, logging, and rate limiting are handled at the server layer

**MCP server structure:**
```
MCP Server
├── list_tools()       → returns available tool schemas
├── call_tool(name, args) → executes a tool and returns result
├── list_resources()   → exposes data resources (files, databases)
└── list_prompts()     → exposes reusable prompt templates
```
</small>

### 3.1 Building a Local MCP Server
<small>We write a finance + education MCP server and save it to `mcp_c24_server.py`.
The server exposes four tools: stock price lookup, portfolio analysis, course search, and enrollment check.</small>

In [ ]:
MCP_SERVER_CODE = '''
import asyncio, json
from mcp.server.fastmcp import FastMCP

mcp = FastMCP('finance-education-tools')

# ── Mock data ────────────────────────────────────────────────────────────────
STOCK_DATA = {
    'AAPL': {'price': 182.50, 'change_pct': 1.2,  'sector': 'Technology'},
    'JPM' : {'price': 198.75, 'change_pct': -0.3, 'sector': 'Finance'},
    'AMZN': {'price': 175.40, 'change_pct': 0.8,  'sector': 'E-commerce'},
    'UNH' : {'price': 512.30, 'change_pct': -1.1, 'sector': 'Healthcare'},
    'EDU' : {'price': 45.20,  'change_pct': 2.1,  'sector': 'Education'},
}

COURSES = {
    'DS-101': {'title': 'Intro to Python',   'prereqs': [],        'seats_left': 45},
    'DS-201': {'title': 'Stats for DS',      'prereqs': ['DS-101'],'seats_left': 12},
    'DS-301': {'title': 'ML Fundamentals',   'prereqs': ['DS-201'],'seats_left': 8},
    'FIN-201': {'title': 'Corporate Finance','prereqs': [],        'seats_left': 30},
}

# ── Tools ────────────────────────────────────────────────────────────────────
@mcp.tool()
def get_stock_price(ticker: str) -> str:
    \"\"\"Return current price and sector for a stock ticker.\"\"\"
    t = ticker.upper()
    if t not in STOCK_DATA:
        return json.dumps({'error': f'Ticker {t} not found'})
    d = STOCK_DATA[t]
    return json.dumps({'ticker': t, **d})

@mcp.tool()
def get_portfolio_summary(tickers: list[str]) -> str:
    \"\"\"Summarise a portfolio given a list of ticker symbols.\"\"\"
    holdings = []
    for t in tickers:
        d = STOCK_DATA.get(t.upper())
        if d:
            holdings.append({'ticker': t.upper(), **d})
    if not holdings:
        return json.dumps({'error': 'No valid tickers found'})
    sectors = {}
    for h in holdings:
        sectors[h[\'sector\']] = sectors.get(h[\'sector\'], 0) + 1
    return json.dumps({'holdings': holdings, 'sector_weights': sectors})

@mcp.tool()
def search_courses(topic: str, level: str = \'any\') -> str:
    \"\"\"Search available courses by topic keyword.\"\"\"
    matches = []
    for cid, info in COURSES.items():
        if topic.lower() in info[\'title\'].lower() or topic.lower() in cid.lower():
            matches.append({'id': cid, **info})
    return json.dumps({'courses': matches, 'query': topic})

@mcp.tool()
def check_enrollment(course_id: str, student_completed: list[str]) -> str:
    \"\"\"Check if a student is eligible to enroll in a course.\"\"\"
    c = COURSES.get(course_id.upper())
    if not c:
        return json.dumps({'error': f'Course {course_id} not found'})
    missing = [p for p in c[\'prereqs\'] if p not in student_completed]
    eligible = len(missing) == 0
    return json.dumps({
        'course_id': course_id,
        'eligible': eligible,
        'missing_prereqs': missing,
        'seats_left': c[\'seats_left\'],
    })

if __name__ == \'__main__\':
    mcp.run(transport=\'stdio\')
'''

# Save the server to a file
Path('mcp_c24_server.py').write_text(MCP_SERVER_CODE, encoding='utf-8')
print('MCP server saved to mcp_c24_server.py')

### 3.2 MCP Server Structure Explained

<small>
```
@mcp.tool()            ← registers the function as an MCP tool
def get_stock_price(ticker: str) -> str:
    \"\"\"Return current price...\"\"\"   ← docstring becomes the tool description
    ...                ← implementation
```

The `FastMCP` class auto-generates the JSON schema from Python type hints.
The server communicates over **stdio** (default) — the client spawns it as a subprocess.
For network deployment, use `transport='sse'` or `transport='http'`.
</small>

### 3.3 Connecting to the MCP Server

<small>
The MCP Python SDK provides `stdio_client` to connect to a server subprocess.
The client:
1. Spawns the server process
2. Calls `session.initialize()` to handshake
3. Uses `session.list_tools()` and `session.call_tool()` to interact
</small>

In [ ]:
from mcp import ClientSession
from mcp.client.stdio import stdio_client
from mcp import StdioServerParameters

async def demo_mcp_client():
    params = StdioServerParameters(
        command=sys.executable,
        args=['mcp_c24_server.py'],
    )
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # List available tools
            tools_resp = await session.list_tools()
            print('Available MCP tools:')
            for t in tools_resp.tools:
                print(f'  {t.name:25s} — {t.description}')

            # Call a finance tool
            result = await session.call_tool('get_stock_price', {'ticker': 'JPM'})
            data = json.loads(result.content[0].text)
            print(f'\nJPM stock data: {data}')

            # Call an education tool
            result2 = await session.call_tool(
                'check_enrollment',
                {'course_id': 'DS-201', 'student_completed': ['DS-101']}
            )
            enroll = json.loads(result2.content[0].text)
            print(f'DS-201 enrollment check: {enroll}')

            # Portfolio summary
            result3 = await session.call_tool(
                'get_portfolio_summary',
                {'tickers': ['AAPL', 'JPM', 'UNH', 'EDU']}
            )
            portfolio = json.loads(result3.content[0].text)
            print(f'\nPortfolio sector weights: {portfolio["sector_weights"]}')

await demo_mcp_client()

### 3.4 Claude Agent Using MCP Tools
<small>
We pass the MCP tool schemas directly to the Anthropic API.
The agent decides which tools to call; we execute them on the MCP server.
</small>

In [ ]:
import anthropic

async def run_mcp_agent(user_query: str):
    """Run a Claude agent that uses MCP tools."""
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

    params = StdioServerParameters(
        command=sys.executable, args=['mcp_c24_server.py']
    )

    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # Convert MCP tool schemas to Anthropic tool format
            mcp_tools = await session.list_tools()
            anthropic_tools = [
                {
                    'name': t.name,
                    'description': t.description,
                    'input_schema': t.inputSchema,
                }
                for t in mcp_tools.tools
            ]

            messages = [{'role': 'user', 'content': user_query}]

            for _ in range(5):  # max 5 tool calls
                response = client.messages.create(
                    model='claude-haiku-4-5-20251001',
                    max_tokens=1024,
                    tools=anthropic_tools,
                    messages=messages,
                )

                if response.stop_reason == 'end_turn':
                    answer = ' '.join(
                        b.text for b in response.content if hasattr(b, 'text')
                    )
                    print(f'\nFinal answer:\n{answer}')
                    return answer

                # Execute tool calls via MCP
                tool_results = []
                for block in response.content:
                    if hasattr(block, 'type') and block.type == 'tool_use':
                        print(f'  Calling MCP tool: {block.name}({block.input})')
                        res = await session.call_tool(block.name, block.input)
                        tool_results.append({
                            'type': 'tool_result',
                            'tool_use_id': block.id,
                            'content': res.content[0].text,
                        })

                messages.append({'role': 'assistant', 'content': response.content})
                messages.append({'role': 'user',      'content': tool_results})

    return 'Max iterations reached'

# Finance scenario
print('=== Finance Query ===')
await run_mcp_agent(
    'My portfolio has AAPL, JPM, and UNH. Am I too concentrated in any sector?'
)

# Education scenario
print('\n=== Education Query ===')
await run_mcp_agent(
    'I have completed DS-101. Can I enroll in DS-301?'
)

---
## Section 4 · Multi-Agent Orchestration — Coordinator + Specialist Pattern

<small>
**When is multi-agent complexity justified?**

| Situation | Recommendation |
|-----------|---------------|
| Single domain, single task | Use one agent or a chain |
| Multiple domains, independent tasks | Multi-agent: one specialist per domain |
| Tasks that depend on each other | Multi-agent with explicit context passing |
| Parallel independent subtasks | Multi-agent (run specialists concurrently) |

**The coordinator + specialist pattern:**
- **Coordinator** receives the user request, decides which specialists to activate
- **Specialists** are domain-expert agents that return structured results
- **Coordinator** synthesises specialist outputs into a final response

**Keep to two agents for most use cases.** A third agent usually means the coordinator
can be replaced with routing logic.
</small>

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# ── Specialist agent definitions ─────────────────────────────────────────────

def finance_specialist(query: str, context: dict) -> dict:
    """Finance domain specialist — analyses financial queries."""
    system = (
        'You are a senior financial analyst. '
        'Answer the query using only the context provided. '
        'Return a JSON object with keys: answer (str), confidence (high/medium/low), '
        'data_used (list of strings from context).'
    )
    prompt = (
        f'Context provided by coordinator:\n{json.dumps(context, indent=2)}\n\n'
        f'Finance query: {query}'
    )
    response = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=512,
        system=system,
        messages=[{'role': 'user', 'content': prompt}],
    )
    text = response.content[0].text
    try:
        # Extract JSON from the response
        start = text.find('{')
        end   = text.rfind('}') + 1
        return json.loads(text[start:end]) if start != -1 else {'answer': text, 'confidence': 'medium', 'data_used': []}
    except Exception:
        return {'answer': text, 'confidence': 'medium', 'data_used': []}


def education_specialist(query: str, context: dict) -> dict:
    """Education domain specialist — advises on learning paths."""
    system = (
        'You are an academic advisor specialising in data science and finance curricula. '
        'Answer using only the context provided. '
        'Return a JSON object with keys: recommendation (str), prerequisites_met (bool), '
        'next_steps (list of strings).'
    )
    prompt = (
        f'Context provided by coordinator:\n{json.dumps(context, indent=2)}\n\n'
        f'Education query: {query}'
    )
    response = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=512,
        system=system,
        messages=[{'role': 'user', 'content': prompt}],
    )
    text = response.content[0].text
    try:
        start = text.find('{')
        end   = text.rfind('}') + 1
        return json.loads(text[start:end]) if start != -1 else {'recommendation': text, 'prerequisites_met': None, 'next_steps': []}
    except Exception:
        return {'recommendation': text, 'prerequisites_met': None, 'next_steps': []}

print('Specialist agents defined.')

In [ ]:
# ── Coordinator agent ────────────────────────────────────────────────────────

COORDINATOR_TOOLS = [
    {
        'name': 'call_finance_specialist',
        'description': 'Delegate a finance-related sub-question to the finance specialist. '
                       'Provide any relevant financial data in the context field.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'sub_query': {'type': 'string', 'description': 'The specific finance question'},
                'context':   {'type': 'object',  'description': 'Relevant data to pass to the specialist'},
            },
            'required': ['sub_query', 'context'],
        },
    },
    {
        'name': 'call_education_specialist',
        'description': 'Delegate an education or course-planning sub-question to the education specialist.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'sub_query': {'type': 'string'},
                'context':   {'type': 'object'},
            },
            'required': ['sub_query', 'context'],
        },
    },
]

COORDINATOR_SYSTEM = '''You are a coordinator agent for a professional learning and investment platform.
Your users are finance professionals who also want to upskill via courses.

When a user sends a query:
1. Decide which specialist(s) can best answer it.
2. Call the relevant specialist(s) with a precise sub-query and any useful context.
3. Synthesise their answers into a concise, actionable response for the user.

Do not guess — if you need financial or educational data, ask the right specialist.
'''

def run_coordinator(user_query: str, shared_context: dict) -> str:
    """Coordinator that orchestrates finance and education specialists."""
    messages = [{
        'role': 'user',
        'content': f'User context: {json.dumps(shared_context)}\n\nUser query: {user_query}',
    }]

    coord_log = []

    for step in range(5):
        response = client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=1024,
            system=COORDINATOR_SYSTEM,
            tools=COORDINATOR_TOOLS,
            messages=messages,
        )

        if response.stop_reason == 'end_turn':
            final = ' '.join(b.text for b in response.content if hasattr(b, 'text'))
            return final, coord_log

        tool_results = []
        for block in response.content:
            if not (hasattr(block, 'type') and block.type == 'tool_use'):
                continue

            tool_name = block.name
            sub_query = block.input.get('sub_query', '')
            ctx       = block.input.get('context', {})

            # Route to the correct specialist
            if tool_name == 'call_finance_specialist':
                specialist_result = finance_specialist(sub_query, ctx)
                coord_log.append({'step': step, 'specialist': 'finance', 'query': sub_query})
            elif tool_name == 'call_education_specialist':
                specialist_result = education_specialist(sub_query, ctx)
                coord_log.append({'step': step, 'specialist': 'education', 'query': sub_query})
            else:
                specialist_result = {'error': f'Unknown tool {tool_name}'}

            tool_results.append({
                'type': 'tool_result',
                'tool_use_id': block.id,
                'content': json.dumps(specialist_result),
            })

        messages.append({'role': 'assistant', 'content': response.content})
        messages.append({'role': 'user',      'content': tool_results})

    return 'Max steps reached', coord_log

print('Coordinator agent defined.')

In [ ]:
# ── Demo: user is a finance professional who wants to expand into AI ─────────

shared_context = {
    'user_profile': {
        'role': 'Portfolio Manager',
        'completed_courses': ['DS-101'],
        'portfolio_tickers': ['AAPL', 'JPM', 'UNH'],
        'risk_tolerance': 'moderate',
    }
}

query = (
    'Given my portfolio and my DS-101 background, '
    'should I invest in healthcare AI stocks '
    'and which course should I take next to better evaluate AI companies?'
)

print(f'User query: {query}\n')
final_answer, log = run_coordinator(query, shared_context)

print('=== Coordinator log ===')
for entry in log:
    print(f"  Step {entry['step']}: delegated to {entry['specialist']} specialist")
    print(f"    Sub-query: {entry['query']}")

print(f'\n=== Final answer ===\n{final_answer}')

---
## Section 5 · Prompt Versioning and A/B Testing

<small>
In production, prompts are as important as code.
A prompt change that improves accuracy by 5% can be worth more than a week of engineering.

**Why version prompts?**
- Roll back quickly if a new prompt regresses
- Track which prompt version produced which output
- Run controlled A/B tests with production traffic
- Audit trail for regulated industries (finance, healthcare)

**Prompt registry pattern:**
- Central registry maps `(name, version)` → template
- Each deployment pins a version
- A/B test by randomly assigning users to version A or B
- Measure: output quality score, user satisfaction, task completion rate
</small>

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
import random

@dataclass
class PromptVersion:
    name:       str
    version:    str
    template:   str
    description: str = ''
    author:     str = ''
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())
    tags:       list = field(default_factory=list)
    metrics:    dict = field(default_factory=dict)   # filled by A/B test


class PromptRegistry:
    """Central registry for versioned prompt templates."""

    def __init__(self):
        self._store: dict[str, dict[str, PromptVersion]] = {}

    def register(self, pv: PromptVersion):
        self._store.setdefault(pv.name, {})[pv.version] = pv
        print(f'Registered prompt  name={pv.name}  version={pv.version}')

    def get(self, name: str, version: str = 'latest') -> PromptVersion:
        if name not in self._store:
            raise KeyError(f'Prompt "{name}" not found')
        versions = self._store[name]
        if version == 'latest':
            version = sorted(versions.keys())[-1]
        if version not in versions:
            raise KeyError(f'Version "{version}" not found for "{name}"')
        return versions[version]

    def list_versions(self, name: str) -> list[str]:
        return sorted(self._store.get(name, {}).keys())

    def all_prompts(self) -> list[str]:
        return list(self._store.keys())

    def record_metric(self, name: str, version: str, metric: str, value: float):
        pv = self.get(name, version)
        pv.metrics.setdefault(metric, []).append(value)

    def compare_versions(self, name: str) -> dict:
        result = {}
        for ver, pv in self._store.get(name, {}).items():
            if pv.metrics:
                result[ver] = {
                    k: round(sum(v)/len(v), 3)
                    for k, v in pv.metrics.items()
                }
        return result


registry = PromptRegistry()
print('PromptRegistry ready.')

In [ ]:
# ── Finance: Loan Approval Decision Prompt — two versions ───────────────────

registry.register(PromptVersion(
    name='loan_decision',
    version='v1',
    template=(
        'Evaluate this loan application and respond with APPROVE or DENY.\n'
        'Application: {application}'
    ),
    description='Baseline prompt — simple approve/deny',
    author='risk-team',
    tags=['finance', 'credit'],
))

registry.register(PromptVersion(
    name='loan_decision',
    version='v2',
    template=(
        'You are a senior credit analyst. Evaluate the following loan application.\n'
        'Respond with a JSON object containing:\n'
        '- decision: APPROVE or DENY\n'
        '- risk_score: 1-10 (10 = highest risk)\n'
        '- key_factors: list of up to 3 decisive factors\n'
        '- conditions: any conditions attached to approval (empty list if none)\n\n'
        'Application:\n{application}'
    ),
    description='Structured JSON output with risk scoring',
    author='risk-team',
    tags=['finance', 'credit', 'structured-output'],
))

# ── Education: Course Recommendation Prompt — two versions ───────────────────

registry.register(PromptVersion(
    name='course_recommendation',
    version='v1',
    template='Recommend a course for: {student_profile}',
    description='Simple recommendation',
    author='edu-team',
))

registry.register(PromptVersion(
    name='course_recommendation',
    version='v2',
    template=(
        'You are an academic advisor.\n'
        'Student profile: {student_profile}\n'
        'Available courses: {available_courses}\n\n'
        'Recommend the single best next course. Explain why in 2 sentences.',
    ),
    description='Contextual recommendation with available courses',
    author='edu-team',
))

print('All registered prompts:', registry.all_prompts())
for name in registry.all_prompts():
    print(f'  {name}: versions = {registry.list_versions(name)}')

### 5.1 A/B Testing Framework
<small>
We randomly assign each request to version A or version B, then track quality scores.
After N requests, compare average scores to decide which version wins.
In production, use user-segment based routing (e.g., 10% traffic to v2) for safer rollouts.
</small>

In [ ]:
class ABTest:
    """Simple A/B test runner for prompt versions."""

    def __init__(self, registry: PromptRegistry, prompt_name: str,
                 version_a: str, version_b: str, traffic_split: float = 0.5):
        self.registry      = registry
        self.prompt_name   = prompt_name
        self.version_a     = version_a
        self.version_b     = version_b
        self.traffic_split = traffic_split  # fraction routed to A
        self.results: list[dict] = []

    def assign_version(self, request_id: str) -> str:
        """Deterministic assignment based on request_id hash."""
        h = hash(request_id) % 100
        return self.version_a if h < self.traffic_split * 100 else self.version_b

    def run(self, request_id: str, template_vars: dict, score_fn) -> dict:
        version = self.assign_version(request_id)
        pv      = self.registry.get(self.prompt_name, version)
        prompt  = pv.template.format(**template_vars)

        # Call the LLM
        response = client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=512,
            messages=[{'role': 'user', 'content': prompt}],
        )
        output = response.content[0].text

        # Score the output
        score = score_fn(output)
        self.registry.record_metric(self.prompt_name, version, 'quality_score', score)

        result = {
            'request_id': request_id,
            'version':    version,
            'output':     output[:200],
            'score':      score,
        }
        self.results.append(result)
        return result

    def summary(self) -> dict:
        return self.registry.compare_versions(self.prompt_name)


# ── Loan decision A/B test ───────────────────────────────────────────────────

loan_applications = [
    {'id': 'req-001', 'app': 'Annual income $85,000. Loan amount $200,000. Credit score 740. Employed 4 years.'},
    {'id': 'req-002', 'app': 'Annual income $32,000. Loan amount $150,000. Credit score 580. Self-employed 1 year.'},
    {'id': 'req-003', 'app': 'Annual income $120,000. Loan amount $250,000. Credit score 810. Employed 10 years.'},
    {'id': 'req-004', 'app': 'Annual income $55,000. Loan amount $180,000. Credit score 650. Employed 2 years.'},
]

def score_loan_response(output: str) -> float:
    """Heuristic quality score: higher for structured JSON with all required fields."""
    score = 0.5  # baseline
    if 'decision' in output.lower():  score += 0.15
    if 'risk_score' in output.lower(): score += 0.15
    if 'key_factors' in output.lower(): score += 0.1
    if 'conditions' in output.lower(): score += 0.1
    return round(score, 2)

ab_test = ABTest(registry, 'loan_decision', 'v1', 'v2', traffic_split=0.5)

print('Running A/B test on loan_decision prompt...\n')
for item in loan_applications:
    result = ab_test.run(
        request_id=item['id'],
        template_vars={'application': item['app']},
        score_fn=score_loan_response
    )
    print(f"  [{result['request_id']}] version={result['version']} score={result['score']}")
    print(f"  output: {result['output'][:120]}...\n")

print('=== A/B Test Summary ===')
print(json.dumps(ab_test.summary(), indent=2))

---
## Section 6 · Lab — End-to-End Multi-Step AI Workflow

### Scenario: Investment & Learning Assistant

<small>
A fintech platform serves users who are both **investors** and **learners**.
The assistant must:
1. Answer investment questions using a RAG index over financial documents
2. Recommend courses based on the user's learning history
3. Call live data tools via MCP (stock prices, course enrollment)
4. Manage conversation memory so context accumulates across turns
5. Use versioned prompts so the team can A/B test response quality

**Components used in this lab:**
| Component | Purpose |
|-----------|---------|
| LangChain LCEL chain | Orchestrate prompt → LLM → parser |
| LlamaIndex VectorStoreIndex | RAG over financial documents |
| MCP tools | Live stock prices and enrollment checks |
| Multi-agent coordinator | Route finance vs education sub-tasks |
| PromptRegistry | Version-controlled prompts |
| RunnableWithMessageHistory | Per-user memory |
</small>

### 6.1 Component Setup

In [ ]:
# ── Prompt registry for the lab ──────────────────────────────────────────────
lab_registry = PromptRegistry()

lab_registry.register(PromptVersion(
    name='investment_assistant',
    version='v1',
    template=(
        'You are an investment and learning assistant.\n'
        'Use the retrieved context to answer accurately.\n\n'
        'Context:\n{context}\n\n'
        'Question: {question}'
    ),
    author='lab',
))

lab_registry.register(PromptVersion(
    name='investment_assistant',
    version='v2',
    template=(
        'You are a personalised investment and learning assistant for finance professionals.\n'
        'User profile: {user_profile}\n\n'
        'Relevant context:\n{context}\n\n'
        'Respond in two sections:\n'
        '1. **Investment insight** (if relevant)\n'
        '2. **Learning recommendation** (if relevant)\n\n'
        'Question: {question}'
    ),
    author='lab',
))

print('Lab prompt registry ready. Versions:', lab_registry.list_versions('investment_assistant'))

In [ ]:
# ── LlamaIndex RAG component ─────────────────────────────────────────────────

lab_index = VectorStoreIndex.from_documents(all_docs)  # reuse docs from Section 2
lab_qe    = lab_index.as_query_engine(similarity_top_k=3)

def retrieve_context(question: str) -> str:
    """Retrieve relevant document context for a question."""
    result = lab_qe.query(question)
    return str(result)

print('RAG index ready. Testing retrieval...')
ctx = retrieve_context('What are the growth prospects for edtech companies?')
print(ctx[:300])

In [ ]:
# ── LCEL chain with versioned prompt and memory ──────────────────────────────

def build_lab_chain(prompt_version: str = 'v2'):
    pv = lab_registry.get('investment_assistant', prompt_version)
    prompt_template = ChatPromptTemplate.from_messages([
        ('system', pv.template),
        MessagesPlaceholder(variable_name='history'),
        ('human', '{question}'),
    ])
    return prompt_template | llm | parser

lab_session_store = {}

def get_lab_history(session_id: str):
    if session_id not in lab_session_store:
        lab_session_store[session_id] = InMemoryChatMessageHistory()
    return lab_session_store[session_id]

def build_memory_chain(prompt_version: str = 'v2'):
    base_chain = build_lab_chain(prompt_version)
    return RunnableWithMessageHistory(
        base_chain,
        get_lab_history,
        input_messages_key='question',
        history_messages_key='history',
    )

print('Lab chain factory ready.')

### 6.2 Full Workflow Execution
<small>
The assistant receives a user query, retrieves RAG context, passes it through the versioned chain,
and maintains memory across turns.
</small>

In [ ]:
async def run_lab_workflow(session_id: str, user_query: str,
                           user_profile: dict, prompt_version: str = 'v2',
                           use_mcp: bool = True) -> str:
    """
    Full workflow:
    1. Retrieve RAG context (LlamaIndex)
    2. Call MCP tool if live data is needed
    3. Run LCEL chain with versioned prompt + memory
    """
    # Step 1: RAG retrieval
    rag_context = retrieve_context(user_query)

    # Step 2: MCP live data (optional)
    mcp_data = {}
    if use_mcp and any(k in user_query.lower() for k in ['price', 'stock', 'enroll', 'course']):
        try:
            params = StdioServerParameters(command=sys.executable, args=['mcp_c24_server.py'])
            async with stdio_client(params) as (read, write):
                async with ClientSession(read, write) as session:
                    await session.initialize()
                    # Pull portfolio prices if tickers mentioned
                    for ticker in ['AAPL', 'JPM', 'UNH']:
                        if ticker in user_query.upper():
                            res = await session.call_tool('get_stock_price', {'ticker': ticker})
                            mcp_data[ticker] = json.loads(res.content[0].text)
        except Exception as e:
            mcp_data['mcp_error'] = str(e)

    # Step 3: Build enriched context
    full_context = rag_context
    if mcp_data:
        full_context += f'\n\nLive market data:\n{json.dumps(mcp_data, indent=2)}'

    # Step 4: LCEL chain with memory
    chain = build_memory_chain(prompt_version)
    response = chain.invoke(
        {
            'question':     user_query,
            'context':      full_context,
            'user_profile': json.dumps(user_profile),
        },
        config={'configurable': {'session_id': session_id}}
    )
    return response

print('Lab workflow function ready.')

In [ ]:
# ── Run the lab workflow ─────────────────────────────────────────────────────

student_investor_profile = {
    'name': 'Priya',
    'role': 'Financial Analyst',
    'completed_courses': ['DS-101', 'FIN-201'],
    'portfolio': ['AAPL', 'JPM', 'UNH'],
    'risk_tolerance': 'moderate',
    'learning_goal': 'Evaluate AI-driven healthcare companies',
}

lab_session = 'priya-session-001'

conversation = [
    'What are the growth prospects for healthcare AI companies in my portfolio?',
    'Which course should I take next to better analyse AI companies?',
    'Based on what we discussed, what is the single most important risk I should monitor?',
]

for turn, query in enumerate(conversation, 1):
    print(f'\n[Turn {turn}] Priya: {query}')
    answer = await run_lab_workflow(
        session_id=lab_session,
        user_query=query,
        user_profile=student_investor_profile,
        prompt_version='v2',
        use_mcp=True,
    )
    print(f'[Turn {turn}] Assistant:\n{answer[:500]}')
    print('-' * 60)

In [ ]:
# ── A/B test the lab prompt versions ────────────────────────────────────────

def score_lab_response(output: str) -> float:
    """Score: rewards structured sections and domain-specific language."""
    score = 0.4
    if 'investment' in output.lower(): score += 0.2
    if 'learning' in output.lower() or 'course' in output.lower(): score += 0.2
    if len(output.split()) > 50:  score += 0.2  # minimum depth
    return round(score, 2)

lab_ab = ABTest(lab_registry, 'investment_assistant', 'v1', 'v2', traffic_split=0.5)

test_queries = [
    ('q-001', 'What are the top investment risks for 2024?', {'question': 'What are the top investment risks for 2024?', 'context': 'Healthcare AI market expected 48% CAGR.', 'user_profile': '{}'}),
    ('q-002', 'Which DS course should I take after DS-201?', {'question': 'Which DS course should I take after DS-201?', 'context': 'DS-301 ML Fundamentals requires DS-201.', 'user_profile': '{}'}),
    ('q-003', 'Is EduLearn a good investment?', {'question': 'Is EduLearn a good investment?', 'context': 'EduLearn Q3: revenue +28% YoY, NPS 72.', 'user_profile': '{}'}),
]

print('Running prompt A/B test...\n')
for qid, _, tvars in test_queries:
    result = lab_ab.run(qid, tvars, score_lab_response)
    print(f"  [{qid}] v={result['version']} score={result['score']}")

print('\n=== A/B Test Summary ===')
print(json.dumps(lab_ab.summary(), indent=2))

### 6.3 Lab Extensions

<small>
**Extension 1 — Add a healthcare specialist**
Extend the coordinator in Section 4 with a third specialist that answers clinical questions.
Keep the two-agent rule by making the healthcare specialist a sub-tool of the education specialist.

**Extension 2 — Persistent memory**
Replace `InMemoryChatMessageHistory` with a Redis-backed store using `langchain_community.chat_message_histories.RedisChatMessageHistory`.
Test that memory survives process restarts.

**Extension 3 — Prompt performance dashboard**
Extend `PromptRegistry.compare_versions()` to return confidence intervals (mean ± std).
Add a `PromptRegistry.auto_promote()` method that marks the highest-scoring version as production.

**Extension 4 — LlamaIndex sub-question engine**
Replace the basic VectorStoreIndex query engine with a `SubQuestionQueryEngine` that decomposes
complex questions like 'Compare the growth rates of FinTech Corp and EduLearn Inc.' into sub-queries.

**Extension 5 — MCP server with authentication**
Add an API key header check to the MCP server using the `httpx` transport instead of stdio.
Demonstrate that unauthenticated clients receive an error.
</small>